In [2]:
!pip3 install openmeteo-requests
!pip3 install requests-cache retry-requests numpy pandas

  Obtaining dependency information for openmeteo-requests from https://files.pythonhosted.org/packages/50/1f/61d55b3b0970ba3f601d6775439b8aae75bcfee159c9cc9c61cd27bb588a/openmeteo_requests-1.1.0-py3-none-any.whl.metadata
  Obtaining dependency information for openmeteo-sdk>=1.4.0 from https://files.pythonhosted.org/packages/62/08/47380296c5af07396cc3a18571c692ab2130839bc58d16d4d1b654996674/openmeteo_sdk-1.7.0-py3-none-any.whl.metadata
  Obtaining dependency information for requests from https://files.pythonhosted.org/packages/70/8e/0e2d847013cb52cd35b38c009bb167a1a26b2ce6cd6965bf26b47bc0bf44/requests-2.31.0-py3-none-any.whl.metadata
  Obtaining dependency information for flatbuffers>=23.0.0 from https://files.pythonhosted.org/packages/6f/12/d5c79ee252793ffe845d58a913197bfa02ae9a0b5c9bc3dc4b58d477b9e7/flatbuffers-23.5.26-py2.py3-none-any.whl.metadata
  Obtaining dependency information for charset-normalizer<4,>=2 from https://files.pythonhosted.org/packages/2e/7d/2259318c202f3d17f3fe643

In [1]:
import openmeteo_requests
import requests_cache
import pandas as pd
from retry_requests import retry
import csv
import os

# latitude - longitude
lat=-20.443810
lon=-54.849570
#aaaa-mm-dd - data inicial e final (recomendado máximo até 1 semana antes do dia de hoje)
start_date="2022-12-01"
end_date="2023-12-01"

dicio_period = {"period 1": ["23/12/2022", "20/01/2023"],
                "period 2": ["20/01/2023", "16/02/2023"],
                "period 3": ["16/02/2023", "07/03/2023"],
                "period 4": ["07/03/2023", "21/03/2023"],
                "period 5": ["21/03/2023", "12/04/2023"],
                "period 6": ["12/04/2023", "28/04/2023"],
                "period 7": ["28/04/2023", "11/05/2023"],
                "period 8": ["11/05/2023", "30/05/2023"],
                "period 9": ["30/05/2023", "16/06/2023"],
                "period 10": ["16/06/2023", "11/07/2023"],
                "period 11": ["11/07/2023", "03/08/2023"],
                "period 12": ["03/08/2023", "22/08/2023"],
                "period 13": ["22/08/2023", "15/09/2023"],
                "period 14": ["15/09/2023", "04/10/2023"],
                "period 15": ["04/10/2023", "26/10/2023"]}

In [2]:
# pega dados da API histórica e salva em hourly_data e daily_data
# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
	"latitude": lat,
	"longitude": lon,
	"start_date": start_date,
	"end_date": end_date,
	"hourly": ["relative_humidity_2m", "surface_pressure"],
	"daily": ["temperature_2m_mean", "rain_sum", "shortwave_radiation_sum", "et0_fao_evapotranspiration"],
	"timezone": "auto"
}
responses = openmeteo.weather_api(url, params=params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates {response.Latitude()}°E {response.Longitude()}°N")
print(f"Elevation {response.Elevation()} m asl")
print(f"Timezone {response.Timezone()} {response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0 {response.UtcOffsetSeconds()} s")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_relative_humidity_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_surface_pressure = hourly.Variables(1).ValuesAsNumpy()

hourly_data = {"date": pd.date_range(
	start = pd.to_datetime(hourly.Time(), unit = "s"),
	end = pd.to_datetime(hourly.TimeEnd(), unit = "s"),
	freq = pd.Timedelta(seconds = hourly.Interval()),
	inclusive = "left"
)}
hourly_data["relative_humidity_2m"] = hourly_relative_humidity_2m
hourly_data["surface_pressure"] = hourly_surface_pressure

hourly_dataframe = pd.DataFrame(data = hourly_data)
# print(hourly_dataframe)

# save data to csv
hourly_dataframe.to_csv("hourly_weather_data.csv", index=False)

# Process daily data. The order of variables needs to be the same as requested.
daily = response.Daily()
daily_temperature_2m_mean = daily.Variables(0).ValuesAsNumpy()
daily_rain_sum = daily.Variables(1).ValuesAsNumpy()
daily_shortwave_radiation_sum = daily.Variables(2).ValuesAsNumpy()
daily_et0_fao_evapotranspiration = daily.Variables(3).ValuesAsNumpy()

daily_data = {"date": pd.date_range(
	start = pd.to_datetime(daily.Time(), unit = "s"),
	end = pd.to_datetime(daily.TimeEnd(), unit = "s"),
	freq = pd.Timedelta(seconds = daily.Interval()),
	inclusive = "left"
)}
daily_data["temperature_2m_mean"] = daily_temperature_2m_mean
daily_data["rain_sum"] = daily_rain_sum
daily_data["shortwave_radiation_sum"] = daily_shortwave_radiation_sum
daily_data["et0_fao_evapotranspiration"] = daily_et0_fao_evapotranspiration

daily_dataframe = pd.DataFrame(data = daily_data)
# print(daily_dataframe)

# save data to csv
daily_dataframe.to_csv("daily_weather_data.csv", index=False)

# abre csv hourly_data.csv e cria lista com os dados
with open('hourly_weather_data.csv', newline='') as csvfile:
    reader = csv.DictReader(csvfile)
    dados = []
    for row in reader:
        dados.append(row)

# apaga arquivo hourly_data.csv
os.remove('hourly_weather_data.csv')

# print(dados[0]['date'].split(' ')[0])
# faça uma lista que guardara as medias diarias (data, humidade, pressao)
medias = []
# calcula a media diaria e guarda na lista medias
cont=0
for i in range(len(dados)):
    if i == 0:
        data = dados[i]['date'].split(' ')[0]
        humidade = float(dados[i]['relative_humidity_2m'])
        pressao = float(dados[i]['surface_pressure'])
        cont+=1
    elif dados[i]['date'].split(' ')[0] == data:
        humidade += float(dados[i]['relative_humidity_2m'])
        pressao += float(dados[i]['surface_pressure'])
        cont+=1
    else:
        medias.append([data, humidade/cont, pressao/cont])
        data = dados[i]['date'].split(' ')[0]
        humidade = float(dados[i]['relative_humidity_2m'])
        pressao = float(dados[i]['surface_pressure'])
        cont=1

    if i == len(dados)-1:
        medias.append([data, humidade/cont, pressao/cont])

with open('daily_weather_data.csv', newline='') as csvfile:
    reader = csv.DictReader(csvfile)
    dados = []
    for row in reader:
        dados.append(row)

os.remove('daily_weather_data.csv')

with open('full_daily_weather_data.csv', 'w', newline='') as csvfile:
    fieldnames = ['date', 'temperature_2m_mean', 'rain_sum', 'shortwave_radiation_sum', 'et0_fao_evapotranspiration', 'relative_humidity_2m', 'surface_pressure']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    for i in range(len(dados)):
        for j in range(len(medias)):
            if dados[i]['date'].split(' ')[0] == medias[j][0]:
                # print('ok')
                writer.writerow({'date': dados[i]['date'].split(' ')[0],
                                'temperature_2m_mean': dados[i]['temperature_2m_mean'],
                                'rain_sum': dados[i]['rain_sum'],
                                'shortwave_radiation_sum': dados[i]['shortwave_radiation_sum'],
                                'et0_fao_evapotranspiration': dados[i]['et0_fao_evapotranspiration'],
                                'relative_humidity_2m': medias[j][1], 'surface_pressure': medias[j][2]})
                break

with open('full_daily_weather_data.csv', 'r') as f:
    dicio_meteo = csv.DictReader(f)
    lista_meteo = list(dicio_meteo)
    medias = []
    for dia in dicio_period:
        # pega data no formado dd/mm/aaaa e transforma em aaaa-mm-dd
        data_ini = dicio_period[dia][0].split('/') # ['23', '12', '2022']
        data_ini = data_ini[2]+'-'+data_ini[1]+'-'+data_ini[0]

        data_fim = dicio_period[dia][1].split('/')
        data_fim = data_fim[2]+'-'+data_fim[1]+'-'+data_fim[0]

        # calcula a média de cada coluna entre data_ini e data_fim
        intervalo = False
        for i in range(len(lista_meteo)):
            if data_ini == lista_meteo[i]['date']:
                temperatura = float(lista_meteo[i]['temperature_2m_mean'])
                chuva = float(lista_meteo[i]['rain_sum'])
                radiacao = float(lista_meteo[i]['shortwave_radiation_sum'])
                et0 = float(lista_meteo[i]['et0_fao_evapotranspiration'])
                humidade = float(lista_meteo[i]['relative_humidity_2m'])
                pressao = float(lista_meteo[i]['surface_pressure'])
                cont=1
                intervalo = True

            elif intervalo:
                temperatura += float(lista_meteo[i]['temperature_2m_mean'])
                chuva += float(lista_meteo[i]['rain_sum'])
                radiacao += float(lista_meteo[i]['shortwave_radiation_sum'])
                et0 += float(lista_meteo[i]['et0_fao_evapotranspiration'])
                humidade += float(lista_meteo[i]['relative_humidity_2m'])
                pressao += float(lista_meteo[i]['surface_pressure'])
                cont+=1

            if data_fim == lista_meteo[i]['date']:
                # converte data para dd/mm/aaaa
                data_ini = data_ini.split('-')
                data_ini = data_ini[2]+'/'+data_ini[1]+'/'+data_ini[0]

                data_fim = data_fim.split('-')
                data_fim = data_fim[2]+'/'+data_fim[1]+'/'+data_fim[0]

                medias.append([data_ini, data_fim, temperatura/cont, chuva, chuva/cont, radiacao, radiacao/cont, et0, humidade/cont, pressao/cont])
                intervalo = False
                break

os.remove('full_daily_weather_data.csv')

Coordinates -20.42179298400879°E -54.86920166015625°N
Elevation 424.0 m asl
Timezone b'America/Campo_Grande' b'-04'
Timezone difference to GMT+0 -14400 s


In [3]:
with open('Table_2_Field_Multispectral_Data.csv', 'r') as f:
    dicio = csv.DictReader(f)
    lista = list(dicio)

for i in lista:
    for j in dicio_period:
        if i['PERIOD'] == j:
            i['PERIOD'] = dicio_period[j]

medias2 = []
cont=0
for i in range(0, len(lista)):
    if i == 0:
        data_i = lista[i]['PERIOD'][0]
        data_f = lista[i]['PERIOD'][1]
        B1 = float(lista[i]['BAND1'])
        B2 = float(lista[i]['BAND2'])
        B3 = float(lista[i]['BAND3'])
        B4 = float(lista[i]['BAND4'])
        B5 = float(lista[i]['BAND5'])
        B6 = float(lista[i]['BAND6'])
        B7 = float(lista[i]['BAND7'])
        B8 = float(lista[i]['BAND8'])
        B8A = float(lista[i]['BAND8A'])
        B9 = float(lista[i]['BAND9'])
        B11 = float(lista[i]['BAND11'])
        B12 = float(lista[i]['BAND12'])
        NDVI = float(lista[i]['NDVI_INDEX'])
        NDWI = float(lista[i]['NDWI_INDEX'])
        EVI = float(lista[i]['EVI_INDEX'])
        LAI = float(lista[i]['LAI_INDEX'])
        DVI = float(lista[i]['DVI_INDEX'])
        GCI = float(lista[i]['GCI_INDEX'])
        GEMI = float(lista[i]['GEMI_INDEX'])
        SAVI = float(lista[i]['SAVI_INDEX'])
        cont = 1
    elif lista[i]['PERIOD'][0] == data_i:
        B1 += float(lista[i]['BAND1'])
        B2 += float(lista[i]['BAND2'])
        B3 += float(lista[i]['BAND3'])
        B4 += float(lista[i]['BAND4'])
        B5 += float(lista[i]['BAND5'])
        B6 += float(lista[i]['BAND6'])
        B7 += float(lista[i]['BAND7'])
        B8 += float(lista[i]['BAND8'])
        B8A += float(lista[i]['BAND8A'])
        B9 += float(lista[i]['BAND9'])
        B11 += float(lista[i]['BAND11'])
        B12 += float(lista[i]['BAND12'])
        NDVI += float(lista[i]['NDVI_INDEX'])
        NDWI += float(lista[i]['NDWI_INDEX'])
        EVI += float(lista[i]['EVI_INDEX'])
        LAI += float(lista[i]['LAI_INDEX'])
        DVI += float(lista[i]['DVI_INDEX'])
        GCI += float(lista[i]['GCI_INDEX'])
        GEMI += float(lista[i]['GEMI_INDEX'])
        SAVI += float(lista[i]['SAVI_INDEX'])
        cont+=1
    else:
        medias2.append([data_i, data_f, B1/cont, B2/cont, B3/cont, B4/cont, B5/cont, B6/cont, B7/cont, B8/cont, B8A/cont, B9/cont, B11/cont, B12/cont, NDVI/cont, NDWI/cont, EVI/cont, LAI/cont, DVI/cont, GCI/cont, GEMI/cont, SAVI/cont])
        data_i = lista[i]['PERIOD'][0]
        data_f = lista[i]['PERIOD'][1]
        B1 = float(lista[i]['BAND1'])
        B2 = float(lista[i]['BAND2'])
        B3 = float(lista[i]['BAND3'])
        B4 = float(lista[i]['BAND4'])
        B5 = float(lista[i]['BAND5'])
        B6 = float(lista[i]['BAND6'])
        B7 = float(lista[i]['BAND7'])
        B8 = float(lista[i]['BAND8'])
        B8A = float(lista[i]['BAND8A'])
        B9 = float(lista[i]['BAND9'])
        B11 = float(lista[i]['BAND11'])
        B12 = float(lista[i]['BAND12'])
        NDVI = float(lista[i]['NDVI_INDEX'])
        NDWI = float(lista[i]['NDWI_INDEX'])
        EVI = float(lista[i]['EVI_INDEX'])
        LAI = float(lista[i]['LAI_INDEX'])
        DVI = float(lista[i]['DVI_INDEX'])
        GCI = float(lista[i]['GCI_INDEX'])
        GEMI = float(lista[i]['GEMI_INDEX'])
        SAVI = float(lista[i]['SAVI_INDEX'])
        cont = 1
        # break
    
    if i == len(lista)-1:
        medias2.append([data_i, data_f, B1/cont, B2/cont, B3/cont, B4/cont, B5/cont, B6/cont, B7/cont, B8/cont, B8A/cont, B9/cont, B11/cont, B12/cont, NDVI/cont, NDWI/cont, EVI/cont, LAI/cont, DVI/cont, GCI/cont, GEMI/cont, SAVI/cont])

header2 = ['START_DATE', 'END_DATE', 'BAND1','BAND2','BAND3','BAND4','BAND5','BAND6','BAND7','BAND8','BAND8A','BAND9','BAND11','BAND12','NDVI_INDEX','NDWI_INDEX',
        'EVI_INDEX','LAI_INDEX','DVI_INDEX','GCI_INDEX','GEMI_INDEX','SAVI_INDEX']

with open('average_efeeder_paddock_indexes.csv', 'w') as f:
    writer = csv.writer(f)
    writer.writerow(header2)
    writer.writerows(medias2)

with open('Table_1_Supplementation_Data.csv', 'r') as f:
    dicio = csv.DictReader(f)
    lista = list(dicio)

with open('average_efeeder_paddock_indexes.csv', 'r') as f:
    dicio = csv.DictReader(f)
    medias2 = list(dicio)
os.remove('average_efeeder_paddock_indexes.csv')

for item in lista:
    for media in medias2:
        date_c = media['START_DATE'].split('/')
        date_c = date_c[2] + '-' + date_c[1] + '-' + date_c[0]
        if date_c == item['START_DATE']:
            item['BAND1'] = media['BAND1']
            item['BAND2'] = media['BAND2']
            item['BAND3'] = media['BAND3']
            item['BAND4'] = media['BAND4']
            item['BAND5'] = media['BAND5']
            item['BAND6'] = media['BAND6']
            item['BAND7'] = media['BAND7']
            item['BAND8'] = media['BAND8']
            item['BAND8A'] = media['BAND8A']
            item['BAND9'] = media['BAND9']
            item['BAND11'] = media['BAND11']
            item['BAND12'] = media['BAND12']
            item['NDVI_INDEX'] = media['NDVI_INDEX']
            item['NDWI_INDEX'] = media['NDWI_INDEX']
            item['EVI_INDEX'] = media['EVI_INDEX']
            item['LAI_INDEX'] = media['LAI_INDEX']
            item['DVI_INDEX'] = media['DVI_INDEX']
            item['GCI_INDEX'] = media['GCI_INDEX']
            item['GEMI_INDEX'] = media['GEMI_INDEX']
            item['SAVI_INDEX'] = media['SAVI_INDEX']
            break

# remove outlier animals
for item in lista:
    if item['ANIMAL'] == '10' or item['ANIMAL'] == '28':
        lista.remove(item)

print(lista[0])

print('Done!')

{'START_DATE': '2022-12-23', 'FINAL_DATE': '2023-01-20', 'PERIOD': '1', 'AMOUNT_DAYS': '28', 'ANIMAL': '1', 'START_WEIGHT': '300', 'FINAL_WEIGHT': '310', 'ADG': '0,357', 'ADS': '200.0', 'SUP_TIME': '6, 12, 18', 'SUP_TOTAL': '5200.0', 'TOTAL_TIME': '908', 'TOTAL_ATTENDANCE': '22', 'TOTAL_DAYS': '9', 'TOTAL_TIME (6-12h)': '21', 'TOTAL_ATTENDANCE (6-12h)': '4', 'TOTAL_DAYS (6-12h)': '3', 'TOTAL_TIME (12-18h)': '704', 'TOTAL_ATTENDANCE (12-18h)': '13', 'TOTAL_DAYS (12-18h)': '2', 'TOTAL_TIME (18-00h)': '183', 'TOTAL_ATTENDANCE (18-00h)': '5', 'TOTAL_DAYS (18-00h)': '4', 'TOTAL_TIME (6-8h)': '0', 'TOTAL_ATTENDANCE (6-8h)': '0', 'TOTAL_DAYS (6-8h)': '0', 'TOTAL_TIME (12-14h)': '602', 'TOTAL_ATTENDANCE (12-14h)': '12', 'TOTAL_DAYS (12-14h)': '2', 'TOTAL_TIME (18-20h)': '62', 'TOTAL_ATTENDANCE (18-20h)': '2', 'TOTAL_DAYS (18-20h)': '2', 'TOTAL_TIME_SPRING (09/23-12/21)': '0', 'TOTAL_ATTENDANCE_SPRING (09/23-12/21)': '0', 'TOTAL_DAYS_SPRING (09/23-12/21)': '0', 'TOTAL_TIME_SUMMER (12/22-03/20)'

In [4]:
header = ['START_DATE','FINAL_DATE','PERIOD','AMOUNT_DAYS','ANIMAL','START_WEIGHT','FINAL_WEIGHT','ADG','ADS','SUP_TIME',
          'SUP_TOTAL','TOTAL_TIME','TOTAL_ATTENDANCE','TOTAL_DAYS','TOTAL_TIME (6-12h)','TOTAL_ATTENDANCE (6-12h)',
          'TOTAL_DAYS (6-12h)','TOTAL_TIME (12-18h)','TOTAL_ATTENDANCE (12-18h)','TOTAL_DAYS (12-18h)','TOTAL_TIME (18-00h)',
          'TOTAL_ATTENDANCE (18-00h)','TOTAL_DAYS (18-00h)','TOTAL_TIME (6-8h)','TOTAL_ATTENDANCE (6-8h)','TOTAL_DAYS (6-8h)',
          'TOTAL_TIME (12-14h)','TOTAL_ATTENDANCE (12-14h)','TOTAL_DAYS (12-14h)','TOTAL_TIME (18-20h)',
          'TOTAL_ATTENDANCE (18-20h)','TOTAL_DAYS (18-20h)','TOTAL_TIME_SPRING (09/23-12/21)',
          'TOTAL_ATTENDANCE_SPRING (09/23-12/21)','TOTAL_DAYS_SPRING (09/23-12/21)','TOTAL_TIME_SUMMER (12/22-03/20)',
          'TOTAL_ATTENDANCE_SUMMER (12/22-03/20)','TOTAL_DAYS_SUMMER (12/22-03/20)','TOTAL_TIME_AUTUMN (03/23-06/21)',
          'TOTAL_ATTENDANCE_AUTUMN (03/23-06/21)','TOTAL_DAYS_AUTUMN (03/23-06/21)','TOTAL_TIME_WINTER (06/22-09/22)',
          'TOTAL_ATTENDANCE_WINTER (06/22-09/22)','TOTAL_DAYS_WINTER (06/22-09/22)', 'BAND1','BAND2','BAND3','BAND4','BAND5','BAND6','BAND7',
          'BAND8','BAND8A','BAND9','BAND11','BAND12','NDVI_INDEX','NDWI_INDEX','EVI_INDEX',
          'LAI_INDEX','DVI_INDEX','GCI_INDEX','GEMI_INDEX','SAVI_INDEX',
          'TEMP_AVG','RAIN_SUM','RAIN_AVG','RAD_SOL_SUM','RAD_SOL_AVG','EVAPOT','HUM_REL_AVG','PRES_ATM_AVG']


with open('Complete_Dataset.csv', 'w') as f:
  for i in lista:
    for j in medias:
      dt_temp = j[0].split('/')
      dt_temp = dt_temp[2]+'-'+dt_temp[1]+'-'+dt_temp[0]
      if dt_temp == i['START_DATE']:
        i['TEMP_AVG'] = j[2]
        i['RAIN_SUM'] = j[3]
        i['RAIN_AVG'] = j[4]
        i['RAD_SOL_SUM'] = j[5]
        i['RAD_SOL_AVG'] = j[6]
        i['EVAPOT'] = j[7]
        i['HUM_REL_AVG'] = j[8]
        i['PRES_ATM_AVG'] = j[9]
        break

  # percorra a lista, se ANIMAL for 10 ou 28, delete
  for item in lista:
    if item['ANIMAL'] == '10' or item['ANIMAL'] == '28':
      lista.remove(item)

  writer = csv.DictWriter(f, fieldnames=header)
  writer.writeheader()
  writer.writerows(lista)